In [1]:
!pip uninstall -y tensorflow tensorflow-intel tensorflow-text keras keras-nlp
!pip install -q tensorflow==2.17.0 keras==3.5.0 keras-nlp==0.5.1

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: tensorflow-text 2.18.1
Uninstalling tensorflow-text-2.18.1:
  Successfully uninstalled tensorflow-text-2.18.1
Found existing installation: keras 3.5.0
Uninstalling keras-3.5.0:
  Successfully uninstalled keras-3.5.0
Found existing installation: keras-nlp 0.18.1
Uninstalling keras-nlp-0.18.1:
  Successfully uninstalled keras-nlp-0.18.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.3/601.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.1/527.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 99.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

In [2]:
!pip install -q evaluate nltk bert-score rouge_score
!pip install -q keras-nlp matplotlib

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency 

In [3]:
import keras_nlp
import tensorflow as tf
import random
from tensorflow import keras

2025-04-14 16:34:43.866636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-14 16:34:43.888253: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-14 16:34:43.894824: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
import pandas as pd
import evaluate
import nltk
from bert_score import score as bertscore
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import evaluate
rouge = evaluate.load("rouge")

nltk.download("punkt")
smoothie = SmoothingFunction().method4

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
# Load QA Model
qa_model = keras.models.load_model("/kaggle/input/gpt3_qa_medchatbot_v2/keras/default/1/fine_tuned_gpt2_qa_v2.keras", compile=False)
qa_model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-5))

# Load Dialog Model
dialog_model = keras.models.load_model("/kaggle/input/gpt2_medchatbot_dialog_v4/keras/default/1/fine_tuned_gpt2_dialog_v4.keras", compile=False)
dialog_model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-5))

I0000 00:00:1744648508.256601      19 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1744648508.257118      19 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 394 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [6]:
# Load dataset
df = pd.read_csv("/kaggle/input/medquad-medical-question-answer-for-ai-research/medquad.csv")

df["text"] = df.apply(
    lambda row: f"Patient: {row['question']}\nDoctor: {row['answer']}", axis=1
)

In [7]:
# Create a list to hold test results
results = []

# Generate and collect answers for 10 random questions
sample_df = df.sample(1000, random_state=42)[["question", "answer"]]

for i, row in sample_df.iterrows():
    question = row["question"]
    true_answer = row["answer"]
    prompt = f"Patient: {question}\nDoctor:"
    
    # Generate model response
    generated = qa_model.generate(prompt, max_length=250)
    
    # Save to results
    results.append({
        "question": question,
        "true_answer": true_answer,
        "generated_answer": generated
    })
qa_model_results_df = pd.DataFrame(results)

I0000 00:00:1744648548.716063      98 service.cc:146] XLA service 0x7b5a60014fc0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744648548.716121      98 service.cc:154]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1744648548.725276      98 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [8]:
# Create a list to hold test results
results = []

for i, row in sample_df.iterrows():
    question = row["question"]
    true_answer = row["answer"]
    prompt = f"Patient: {question}\nDoctor:"
    
    # Generate model response
    generated = dialog_model.generate(prompt, max_length=250)
    
    # Save to results
    results.append({
        "question": question,
        "true_answer": true_answer,
        "generated_answer": generated
    })
dialog_model_results_df = pd.DataFrame(results)

In [9]:
# Convert text columns to list (cleaned strings)
qa_preds = qa_model_results_df["generated_answer"].astype(str).tolist()
qa_refs = qa_model_results_df["true_answer"].astype(str).tolist()

dialog_preds = dialog_model_results_df["generated_answer"].astype(str).tolist()
dialog_refs = dialog_model_results_df["true_answer"].astype(str).tolist()

In [10]:
bleu_metric = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# QA model
qa_bleu = bleu_metric.compute(predictions=qa_preds, references=qa_refs)["bleu"]
qa_rouge = rouge.compute(predictions=qa_preds, references=qa_refs)

# Dialog model
dialog_bleu = bleu_metric.compute(predictions=dialog_preds, references=dialog_refs)["bleu"]
dialog_rouge = rouge.compute(predictions=dialog_preds, references=dialog_refs)

In [11]:
# QA model
_, _, qa_bert_f1 = bertscore(qa_preds, qa_refs, lang="en", verbose=False)

# Dialog model
_, _, dialog_bert_f1 = bertscore(dialog_preds, dialog_refs, lang="en", verbose=False)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
evaluation_df = pd.DataFrame({
    "BLEU": [qa_bleu, dialog_bleu],
    "ROUGE-1": [qa_rouge["rouge1"], dialog_rouge["rouge1"]],
    "ROUGE-L": [qa_rouge["rougeL"], dialog_rouge["rougeL"]],
    "BERTScore (F1)": [qa_bert_f1.mean().item(), dialog_bert_f1.mean().item()]
}, index=["QA Model", "Dialog Model"])

display(evaluation_df)

,BLEU,ROUGE-1,ROUGE-L,BERTScore (F1)
QA Model,0.057224,0.293736,0.180353,0.839159
Dialog Model,0.012499,0.184406,0.117547,0.812329


In [13]:
evaluation_df.to_csv("/kaggle/working/model_comparison_scores.csv")